In [ ]:
# -*- coding: utf-8 -*-
"""co_so_AI.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1IWWJ1g7l44n_2Cy9tvFGa88AL6V3vgB2
"""

EMPTY = " "

X = "X"
O = "O"
WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]

MOVE_PRIORITY = [4, 0, 2, 6, 8, 1, 3, 5, 7]

def render_cell(board, index):
    if board[index] == EMPTY:
        return str(index + 1)
    return board[index]

def print_board(board):
    print()
    print(f" {render_cell(board, 0)} | {render_cell(board, 1)} | {render_cell(board, 2)} ")
    print("---+---+---")
    print(f" {render_cell(board, 3)} | {render_cell(board, 4)} | {render_cell(board, 5)} ")
    print("---+---+---")
    print(f" {render_cell(board, 6)} | {render_cell(board, 7)} | {render_cell(board, 8)} ")
    print()

def format_move(move):
    row = move // 3 + 1
    col = move % 3 + 1
    return f"ô {move + 1} (hàng {row}, cột {col}, index = {move})"

def check_winner(board):
    for a, b, c in WIN_LINES:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    if EMPTY not in board:
        return "Draw"
    return None

def utility(board):
    result = check_winner(board)

    if result == X:
        return 1
    elif result == O:
        return -1
    elif result == "Draw":
        return 0
    else:
        return None


def get_available_moves(board):
    return [i for i in range(9) if board[i] == EMPTY]

def ordered_moves(board):
    available = set(get_available_moves(board))
    return [move for move in MOVE_PRIORITY if move in available]


def make_move(board, move, player):
    new_board = board.copy()
    new_board[move] = player
    return new_board

def next_player(player):
    if player == X:
        return O
    return X

from functools import lru_cache

@lru_cache(maxsize=None)
def minimax(board_tuple, current_player):
    board = list(board_tuple)
    score = utility(board)
    if score is not None:
        return score
    if current_player == X:
        best_value = -float("inf")
        for move in ordered_moves(board):
            new_board = make_move(board, move, X)
            value = minimax(tuple(new_board), O)
            best_value = max(best_value, value)
        return best_value
    else:
        best_value = float("inf")

        for move in ordered_moves(board):
            new_board = make_move(board, move, O)
            value = minimax(tuple(new_board), X)
            best_value = min(best_value, value)
        return best_value

def find_best_move(board, player):
    moves = ordered_moves(board)
    if len(moves) == 0:
        return None, utility(board)
    best_move = None
    if player == X:
        best_value = -float("inf")

        for move in moves:
            new_board = make_move(board, move, X)
            value = minimax(tuple(new_board), O)

            if value > best_value:
                best_value = value
                best_move = move

    else:
        best_value = float("inf")

        for move in moves:
            new_board = make_move(board, move, O)
            value = minimax(tuple(new_board), X)

            if value < best_value:
                best_value = value
                best_move = move

    return best_move, best_value


def print_candidate_moves(board, player):
    print(f"Các nước đi có thể của {player}:")

    for move in ordered_moves(board):
        new_board = make_move(board, move, player)
        value = minimax(tuple(new_board), next_player(player))
        print(f"- {format_move(move)} => minimax = {value}")

def run_experiment(name, board, player):
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(f"Lượt hiện tại: {player}")
    print_board(board)
    result = check_winner(board)
    if result is not None:
        print("Trạng thái đã kết thúc.")
        print("Kết quả:", result)
        print("Utility:", utility(board))
        print()
        return

    print_candidate_moves(board, player)

    best_move, best_value = find_best_move(board, player)

    print()
    print("Nước đi tốt nhất:", format_move(best_move))
    print("Giá trị minimax:", best_value)
    print()
board_1 = [EMPTY] * 9

run_experiment(
    name="THỰC NGHIỆM 1: Bàn cờ ban đầu, X/MAX đi trước",
    board=board_1,
    player=X
)
board_2 = [
    X, O, X,
    O, X, EMPTY,
    EMPTY, EMPTY, O
]

run_experiment(
    name="THỰC NGHIỆM 2: X/MAX có nước thắng ngay",
    board=board_2,
    player=X
)
board_3 = [
    O, O, EMPTY,
    X, EMPTY, EMPTY,
    EMPTY, X, EMPTY
]

run_experiment(
    name="THỰC NGHIỆM 3: X/MAX cần chặn O/MIN",
    board=board_3,
    player=X
)

board_4 = [
    O, O, EMPTY,
    X, X, EMPTY,
    EMPTY, EMPTY, EMPTY
]

run_experiment(
    name="THỰC NGHIỆM 4: O/MIN có nước thắng ngay",
    board=board_4,
    player=O
)

def announce_result(board):
    result = check_winner(board)

    if result == X:
        print("Kết quả: X thắng.")
        print("Utility = +1")
    elif result == O:
        print("Kết quả: O thắng.")
        print("Utility = -1")
    elif result == "Draw":
        print("Kết quả: Hòa.")
        print("Utility = 0")


def get_human_move(board, human_player):
    while True:
        try:
            user_input = input(f"Bạn ({human_player}) chọn ô từ 1 đến 9: ")
            move = int(user_input) - 1

            if move < 0 or move > 8:
                print("Ô không hợp lệ. Vui lòng nhập số từ 1 đến 9.")
                continue

            if board[move] != EMPTY:
                print("Ô này đã được đánh. Vui lòng chọn ô khác.")
                continue

            return move

        except ValueError:
            print("Vui lòng nhập một số nguyên từ 1 đến 9.")


def choose_human_player():
    while True:
        choice = input("Bạn muốn chơi X hay O? Nhập X hoặc O: ").strip().upper()

        if choice in [X, O]:
            return choice

        print("Lựa chọn không hợp lệ. Vui lòng nhập X hoặc O.")

def play_game():
    board = [EMPTY] * 9

    print("Trò chơi Tic-Tac-Toe với thuật toán Minimax")
    print("Quy ước:")
    print("- X là MAX")
    print("- O là MIN")
    print("- Utility = +1 nếu X thắng, -1 nếu O thắng, 0 nếu hòa")
    print()

    human_player = choose_human_player()
    ai_player = O if human_player == X else X

    print()
    print(f"Bạn chơi: {human_player}")
    print(f"Máy chơi: {ai_player}")
    print("X luôn đi trước.")
    print()

    current_player = X

    while True:
        print_board(board)

        result = check_winner(board)
        if result is not None:
            announce_result(board)
            break

        if current_player == human_player:
            move = get_human_move(board, human_player)
            board = make_move(board, move, human_player)

        else:
            print(f"Máy ({ai_player}) đang tính bằng Minimax...")

            best_move, best_value = find_best_move(board, ai_player)

            print("Nước đi tốt nhất của máy:", format_move(best_move))
            print("Giá trị minimax:", best_value)
            print()

            board = make_move(board, best_move, ai_player)

        current_player = next_player(current_player)

play_game()